In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# Find project root
ROOT_DIR = Path.cwd()

if not (ROOT_DIR / "data" / "fake_job_postings.csv").exists():
    ROOT_DIR = ROOT_DIR.parent

DATA_PATH = ROOT_DIR / "data" / "fake_job_postings.csv"

# Load dataset
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Loaded successfully.")

Dataset shape: (17880, 18)
Loaded successfully.


In [2]:
# Create combined text exactly as used in our models

text_columns = [
    "title",
    "company_profile",
    "description",
    "requirements",
    "benefits"
]

df["combined_text"] = (
    df[text_columns]
    .fillna("")
    .agg(" ".join, axis=1)
)

# Same train/test split used earlier
X = df["combined_text"]
y = df["fraudulent"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))
print("\nTraining fraud cases:", y_train.sum())
print("Test fraud cases:", y_test.sum())

Training samples: 14304
Test samples: 3576

Training fraud cases: 693
Test fraud cases: 173


In [3]:
vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (14304, 50000)
Test TF-IDF shape: (3576, 50000)


In [4]:
# Logistic Regression
model_lr_eval = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

model_lr_eval.fit(X_train_tfidf, y_train)

# Linear SVM
model_svm_eval = LinearSVC(
    class_weight="balanced",
    random_state=42
)

model_svm_eval.fit(X_train_tfidf, y_train)

print("Both models trained successfully.")

Both models trained successfully.


In [5]:
# Predictions

y_pred_lr = model_lr_eval.predict(X_test_tfidf)
y_pred_svm = model_svm_eval.predict(X_test_tfidf)

# Metrics function
def evaluate_model(name, y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f"\n{name}")
    print("-" * 40)
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))


evaluate_model(
    "Logistic Regression",
    y_test,
    y_pred_lr
)

evaluate_model(
    "Linear SVM",
    y_test,
    y_pred_svm
)


Logistic Regression
----------------------------------------
Accuracy : 0.9835
Precision: 0.7767
Recall   : 0.9249
F1 Score : 0.8443

Confusion Matrix:
[[3357   46]
 [  13  160]]

Linear SVM
----------------------------------------
Accuracy : 0.9913
Precision: 0.9437
Recall   : 0.8728
F1 Score : 0.9069

Confusion Matrix:
[[3394    9]
 [  22  151]]


In [6]:
# Probability/decision scores for PR-AUC

y_prob_lr = model_lr_eval.predict_proba(X_test_tfidf)[:, 1]
y_score_svm = model_svm_eval.decision_function(X_test_tfidf)

pr_auc_lr = average_precision_score(y_test, y_prob_lr)
pr_auc_svm = average_precision_score(y_test, y_score_svm)

print(f"Logistic Regression PR-AUC: {pr_auc_lr:.4f}")
print(f"Linear SVM PR-AUC: {pr_auc_svm:.4f}")

Logistic Regression PR-AUC: 0.9316
Linear SVM PR-AUC: 0.9514


In [7]:
# Create model comparison table

comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Linear SVM",
        "BiLSTM"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_svm),
        0.9807
    ],
    "Precision": [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_svm),
        0.7737
    ],
    "Recall": [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_svm),
        0.8497
    ],
    "F1 Score": [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_svm),
        0.8099
    ],
    "PR-AUC": [
        pr_auc_lr,
        pr_auc_svm,
        0.8738
    ]
})

comparison.round(4)

,Model,Accuracy,Precision,Recall,F1 Score,PR-AUC
0,Logistic Regression,0.9835,0.7767,0.9249,0.8443,0.9316
1,Linear SVM,0.9913,0.9438,0.8728,0.9069,0.9514
2,BiLSTM,0.9807,0.7737,0.8497,0.8099,0.8738


In [8]:
print("Logistic Regression")
print(confusion_matrix(y_test, y_pred_lr))

print("\nLinear SVM")
print(confusion_matrix(y_test, y_pred_svm))

Logistic Regression
[[3357   46]
 [  13  160]]

Linear SVM
[[3394    9]
 [  22  151]]
